# NLP Workshop â€” Notebook 09 (Claude Assisted)
# Advanced RAG Techniques

**Series:** Classical NLP to Modern Embeddings  
**Dataset:** Quran Translations (daryabadi column, 6,236 verses)  
**Prerequisites:** NB08 (RAG and LLM-based Retrieval)

NB08 built a working RAG baseline. Every weakness listed in its "What This Is NOT" section becomes a lesson here. We upgrade the pipeline step by step, measuring the improvement at each stage.

## Roadmap: NB08 vs NB09

| NB08 Baseline | NB09 Upgrade |
|---|---|
| Dense-only retrieval (SBERT) | Hybrid retrieval: BM25 + SBERT fused with RRF |
| No re-ranking | Cross-encoder re-ranking of top-20 candidates |
| Plain query as-is | Query expansion via HyDE |
| Full verse as chunk | Sentence-boundary chunking with overlap |
| Full context passed to LLM | Contextual compression (strip irrelevant sentences) |
| No evaluation | RAGAS-style faithfulness + relevance scoring |
| No comparison | Side-by-side: naive vs advanced on same queries |

## Configuration

All tuneable parameters are defined in one place so you can experiment without hunting through the notebook.

In [ ]:
import re, time, warnings, math
import numpy as np
import pandas as pd
import faiss
import torch
import nltk
from pathlib import Path
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

warnings.filterwarnings("ignore")

for _pkg, _res in [("punkt", "tokenizers/punkt"),
                   ("punkt_tab", "tokenizers/punkt_tab"),
                   ("stopwords", "corpora/stopwords")]:
    try:
        nltk.data.find(_res)
    except LookupError:
        nltk.download(_pkg, quiet=True)

from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords

# --- Config ---
MODEL_NAME     = "all-MiniLM-L6-v2"
CROSS_ENC_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
GEN_MODEL      = "google/flan-t5-small"
USE_LOCAL_LLM  = True
MAX_VERSES     = 2500
TOP_K          = 4
BM25_CANDIDATES = 20    # BM25 recall pool
DENSE_CANDIDATES = 20   # SBERT recall pool
RERANK_POOL    = 20     # candidates fed to cross-encoder
MAX_NEW_TOKENS = 96
NUM_BEAMS      = 2
CONTEXT_MAX_CHARS = 2200
VERSE_MAX_CHARS   = 450
CHUNK_SIZE     = 3      # sentences per chunk
CHUNK_OVERLAP  = 1      # sentences of overlap between chunks
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

## Section 1 â€” Load and Prepare Data

We use the same robust `find_dataset()` path-finder used throughout the series. After loading, we build BM25 token lists by lowercasing, removing punctuation, and stripping stopwords.

In [ ]:
def find_dataset(filename="quran_translations.csv"):
    candidates = [Path("..") / filename, Path(".") / filename, Path("../..")/filename]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError("Could not find " + repr(filename))

CSV_PATH = find_dataset()
df = pd.read_csv(CSV_PATH)[["Surah", "Verse", "daryabadi"]].dropna().reset_index(drop=True)
df["verse_id"] = df["Surah"].astype(str) + ":" + df["Verse"].astype(str)

STOP = set(stopwords.words("english"))
PUNCT_RE = re.compile(r"[^a-z0-9\s']+", re.IGNORECASE)

def tokenize_for_bm25(text):
    text = PUNCT_RE.sub(" ", text.lower())
    return [t for t in word_tokenize(text) if t not in STOP and t.strip()]

df["bm25_tokens"] = df["daryabadi"].apply(tokenize_for_bm25)
df = df[df["bm25_tokens"].str.len() >= 3]
if len(df) > MAX_VERSES:
    df = df.sample(MAX_VERSES, random_state=42).reset_index(drop=True)
print("Verses:", len(df))
df.head(3)

## Section 2 â€” Hybrid Retrieval (BM25 + SBERT)

### Why hybrid?

**BM25** is a sparse keyword-based ranker. It excels when the query uses the exact words that appear in the document â€” great recall for terminology, fast to compute, no GPU needed.

**SBERT** is a dense semantic search. It excels at paraphrases and synonyms because it encodes meaning into a vector space. It can find "compassion" when you query "mercy".

Neither is perfect alone:
- BM25 misses paraphrases ("forgiveness" vs "pardon")
- SBERT can miss rare exact-match terms or proper nouns

Combining them with **Reciprocal Rank Fusion (RRF)** captures the best of both worlds.

### RRF formula

For each document `d` across all ranking systems:

```
score(d) = sum over all systems S of: 1 / (k + rank_S(d))
```

Where `k = 60` is a smoothing constant. Higher-ranked documents get higher scores; the constant `k` prevents top-1 from dominating overwhelmingly.

In [ ]:
# BM25 index
bm25 = BM25Okapi(df["bm25_tokens"].tolist())
print("BM25 index built over", len(df), "verses")

# SBERT index
print("Building SBERT index...")
st_model = SentenceTransformer(MODEL_NAME)
corpus_emb = st_model.encode(
    df["daryabadi"].tolist(),
    batch_size=64,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
).astype(np.float32)
corpus_emb = np.ascontiguousarray(corpus_emb)
d = corpus_emb.shape[1]
index_flat = faiss.IndexFlatIP(d)
index_flat.add(corpus_emb)
print("SBERT FAISS index size:", index_flat.ntotal)

In [ ]:
def rrf_score(rank, k=60):
    return 1.0 / (k + rank)

def hybrid_retrieve(query, top_k=TOP_K, bm25_pool=BM25_CANDIDATES, dense_pool=DENSE_CANDIDATES):
    # BM25 leg
    query_tokens = tokenize_for_bm25(query)
    bm25_scores = bm25.get_scores(query_tokens)
    bm25_top = np.argsort(bm25_scores)[::-1][:bm25_pool]

    # Dense leg
    qvec = np.ascontiguousarray(
        st_model.encode([query], normalize_embeddings=True).astype(np.float32)
    )
    _, dense_top_idx = index_flat.search(qvec, dense_pool)
    dense_top = dense_top_idx[0]

    # RRF fusion
    rrf = {}
    for rank, idx in enumerate(bm25_top, 1):
        rrf[idx] = rrf.get(idx, 0.0) + rrf_score(rank)
    for rank, idx in enumerate(dense_top, 1):
        rrf[idx] = rrf.get(idx, 0.0) + rrf_score(rank)

    ranked = sorted(rrf.items(), key=lambda x: x[1], reverse=True)[:top_k]
    results = []
    for rank, (idx, score) in enumerate(ranked, 1):
        row = df.iloc[idx]
        results.append({"rank": rank, "verse_id": row["verse_id"],
                         "text": row["daryabadi"], "rrf_score": round(score, 5)})
    return results

# Demo
q_demo = "mercy and forgiveness for believers"
hybrid_hits = hybrid_retrieve(q_demo)
print("Hybrid retrieval for:", q_demo)
print()
for h in hybrid_hits:
    print("Rank", h["rank"], " RRF:", h["rrf_score"], " [" + h["verse_id"] + "]")
    print(" ", h["text"][:100])
    print()

In [ ]:
def dense_retrieve(query, k=TOP_K):
    qvec = np.ascontiguousarray(
        st_model.encode([query], normalize_embeddings=True).astype(np.float32)
    )
    scores, indices = index_flat.search(qvec, k)
    return [df.iloc[i]["verse_id"] for i in indices[0]]

dense_ids  = dense_retrieve(q_demo)
hybrid_ids = [h["verse_id"] for h in hybrid_hits]

print("Query:", q_demo)
print()
print("Dense-only top-4 :", dense_ids)
print("Hybrid (RRF) top-4:", hybrid_ids)
overlap = len(set(dense_ids) & set(hybrid_ids))
print("Overlap:", overlap, "of", TOP_K, "verses shared")

## Section 3 â€” Cross-Encoder Re-ranking

### Bi-encoder vs Cross-encoder

A **bi-encoder** (like SBERT) encodes query and document *independently* into vectors and uses cosine similarity to score them. This is fast and scales to millions of documents, but the representations are computed in isolation â€” the model never sees the query and document together.

A **cross-encoder** reads the query and document *jointly* in a single forward pass. It can model complex interactions between them, making it far more accurate. The cost: you cannot pre-compute document vectors, so it is too slow to run on the full corpus.

### Two-stage pipeline (standard in production)

1. **Recall stage**: bi-encoder retrieves top-20 candidates cheaply
2. **Re-rank stage**: cross-encoder scores all 20 pairs, produces the final top-4

This combines the speed of bi-encoders with the accuracy of cross-encoders.

In [ ]:
print("Loading cross-encoder:", CROSS_ENC_NAME)
cross_encoder = CrossEncoder(CROSS_ENC_NAME)
print("Cross-encoder ready")

def rerank(query, candidates, top_k=TOP_K):
    pairs = [(query, c["text"]) for c in candidates]
    ce_scores = cross_encoder.predict(pairs)
    for c, s in zip(candidates, ce_scores):
        c["ce_score"] = float(s)
    reranked = sorted(candidates, key=lambda x: x["ce_score"], reverse=True)[:top_k]
    for rank, c in enumerate(reranked, 1):
        c["rank"] = rank
    return reranked

# Expand pool to 20, then re-rank to top-4
hybrid_pool = hybrid_retrieve(q_demo, top_k=RERANK_POOL)
reranked_hits = rerank(q_demo, hybrid_pool)

print("After re-ranking:", q_demo)
print()
for h in reranked_hits:
    print("Rank", h["rank"], " CE:", round(h["ce_score"], 3), " RRF:", h["rrf_score"],
          " [" + h["verse_id"] + "]")
    print(" ", h["text"][:100])
    print()

In [ ]:
print("Rank shift analysis (RRF rank -> CE rank):")
print()
rrf_rank_map = {h["verse_id"]: h["rank"] for h in hybrid_pool}
for h in reranked_hits:
    vid = h["verse_id"]
    old_rank = rrf_rank_map.get(vid, "?")
    new_rank = h["rank"]
    moved = old_rank - new_rank if isinstance(old_rank, int) else 0
    direction = "^" + str(moved) if moved > 0 else ("v" + str(abs(moved)) if moved < 0 else "=")
    print(vid, "  RRF rank:", old_rank, "-> CE rank:", new_rank, " (" + direction + ")")

## Section 4 â€” Query Expansion via HyDE

### What is HyDE?

**HyDE** = Hypothetical Document Embedding (Gao et al., 2022).

**The problem with short queries**: A query like "mercy for believers" is sparse â€” only a few tokens. The embedding may not capture the full semantic space of relevant documents.

**HyDE's idea**:
1. Use an LLM to generate a *hypothetical answer* to the query â€” a short passage written "in the style" of the corpus
2. Embed that hypothetical document instead of the raw query
3. The hypothetical doc is richer in vocabulary, covers synonyms, and sits closer in embedding space to real relevant documents

**Why it helps**: The generated passage will use terms like "compassion", "sin", "repentance" â€” vocabulary that actually appears in the corpus â€” improving recall.

**When it can fail**:
- The LLM hallucinates facts that don't appear in the corpus at all
- The LLM's writing style is very different from the corpus
- The query is so specific that generic generation adds noise

In [ ]:
if USE_LOCAL_LLM:
    print("Loading generator:", GEN_MODEL)
    gen_tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL)
    gen_model = AutoModelForSeq2SeqLM.from_pretrained(GEN_MODEL).to(DEVICE)
    gen_model.eval()
    print("Generator ready")
else:
    gen_tokenizer = gen_model = None

def hyde_retrieve(query, k=TOP_K):
    if gen_model is None:
        return dense_retrieve(query, k)
    # Step 1: generate a hypothetical answer
    hyde_prompt = "Write a short passage that answers the question: " + query
    inputs = gen_tokenizer(hyde_prompt, return_tensors="pt", truncation=True, max_length=256).to(DEVICE)
    with torch.no_grad():
        out = gen_model.generate(**inputs, max_new_tokens=64, num_beams=2, early_stopping=True)
    hypo_doc = gen_tokenizer.decode(out[0], skip_special_tokens=True)
    print("Hypothetical doc:", hypo_doc[:120])
    # Step 2: embed the hypothetical doc and retrieve
    hypo_vec = np.ascontiguousarray(
        st_model.encode([hypo_doc], normalize_embeddings=True).astype(np.float32)
    )
    scores, indices = index_flat.search(hypo_vec, k)
    results = []
    for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), 1):
        row = df.iloc[idx]
        results.append({"rank": rank, "verse_id": row["verse_id"],
                         "text": row["daryabadi"], "score": float(score)})
    return results

hyde_hits = hyde_retrieve(q_demo)
print()
print("HyDE retrieval for:", q_demo)
for h in hyde_hits:
    print(" [" + h["verse_id"] + "]", h["text"][:90])

In [ ]:
hyde_ids   = [h["verse_id"] for h in hyde_hits]
dense_ids2 = dense_retrieve(q_demo)  # same query for fair comparison

print("Query:", q_demo)
print()
print("--- Dense-only top-" + str(TOP_K) + " ---")
for vid in dense_ids2:
    row = df[df["verse_id"] == vid]
    if len(row) == 0:
        continue
    qvec = np.ascontiguousarray(st_model.encode([q_demo], normalize_embeddings=True).astype(np.float32))
    dvec = np.ascontiguousarray(st_model.encode([row.iloc[0]["daryabadi"]], normalize_embeddings=True).astype(np.float32))
    score = float(np.dot(qvec, dvec.T))
    print(" [", vid, "] score=", round(score, 4), ":", row.iloc[0]["daryabadi"][:80])

print()
print("--- HyDE top-" + str(TOP_K) + " (query embedded as hypothetical doc) ---")
for h in hyde_hits:
    print(" [", h["verse_id"], "] score=", round(h["score"], 4), ":", h["text"][:80])

only_dense  = [v for v in dense_ids2 if v not in hyde_ids]
only_hyde   = [v for v in hyde_ids   if v not in dense_ids2]
shared      = [v for v in dense_ids2 if v in hyde_ids]
print()
print("Shared          :", shared)
print("Dense-only unique:", only_dense, "(missed by HyDE)")
print("HyDE-only unique :", only_hyde,  "(found only via hypothetical doc)")
print()
print("Interpretation: HyDE-only verses were unreachable by the sparse query")
print("but surfaced when the model expanded the query into a richer passage.")

## Section 5 â€” Chunking Strategies

### Why chunk?

NB08 used the full verse as the retrieval unit. This is simple but has two weaknesses:

1. **Long verses** may contain multiple distinct topics. If a verse discusses both "prayer" and "zakat", a query about prayer might retrieve it â€” but the returned text is half irrelevant.
2. **Granularity mismatch**: you want to retrieve the *most relevant sentence*, not the whole verse.

### Sentence-boundary chunking with overlap

- Split each verse into sentences using NLTK's `sent_tokenize`
- Group sentences into chunks of size `CHUNK_SIZE` (e.g. 3 sentences)
- Advance by `CHUNK_SIZE - CHUNK_OVERLAP` sentences per step
- A sentence at a chunk boundary appears in *both* adjacent chunks, ensuring no sentence is orphaned

### Trade-offs

| | Verse-level | Chunk-level |
|---|---|---|
| Index size | Small | Larger (more chunks) |
| Encoding time | Fast | Slower |
| Precision | Lower (noisy) | Higher (focused) |
| Context quality | May include irrelevant sentences | Tighter, more relevant |

In [ ]:
def chunk_text(text, verse_id, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    sentences = sent_tokenize(text)
    chunks = []
    step = max(1, chunk_size - overlap)
    for start in range(0, len(sentences), step):
        chunk_sents = sentences[start:start + chunk_size]
        if len(chunk_sents) == 0:
            break
        chunks.append({
            "verse_id": verse_id,
            "chunk_id": verse_id + "_c" + str(start),
            "text": " ".join(chunk_sents),
        })
    return chunks

# Build chunk corpus from the same df
all_chunks = []
for _, row in df.iterrows():
    all_chunks.extend(chunk_text(row["daryabadi"], row["verse_id"]))

chunks_df = pd.DataFrame(all_chunks)
print("Original verses:", len(df))
print("Chunks created :", len(chunks_df))
print("Avg chunks/verse:", round(len(chunks_df) / len(df), 2))
chunks_df.head(5)

In [ ]:
print("Encoding", len(chunks_df), "chunks...")
chunk_emb = st_model.encode(
    chunks_df["text"].tolist(),
    batch_size=64,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
).astype(np.float32)
chunk_emb = np.ascontiguousarray(chunk_emb)
chunk_index = faiss.IndexFlatIP(chunk_emb.shape[1])
chunk_index.add(chunk_emb)
print("Chunk index size:", chunk_index.ntotal)

def chunk_retrieve(query, k=TOP_K):
    qvec = np.ascontiguousarray(
        st_model.encode([query], normalize_embeddings=True).astype(np.float32)
    )
    scores, indices = chunk_index.search(qvec, k)
    results = []
    for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), 1):
        row = chunks_df.iloc[idx]
        results.append({"rank": rank, "chunk_id": row["chunk_id"],
                         "verse_id": row["verse_id"], "text": row["text"], "score": float(score)})
    return results

chunk_hits = chunk_retrieve(q_demo)
print()
print("Chunk-level retrieval for:", q_demo)
for h in chunk_hits:
    print("Rank", h["rank"], "[" + h["verse_id"] + "]", h["text"][:100])

In [ ]:
# Side-by-side: verse-level retrieval vs chunk-level retrieval
verse_hits = []
for vid, score in zip(
    index_flat.search(
        np.ascontiguousarray(st_model.encode([q_demo], normalize_embeddings=True).astype(np.float32)), TOP_K
    )[1][0],
    index_flat.search(
        np.ascontiguousarray(st_model.encode([q_demo], normalize_embeddings=True).astype(np.float32)), TOP_K
    )[0][0]
):
    row = df.iloc[vid]
    verse_hits.append({"verse_id": row["verse_id"], "text": row["daryabadi"], "score": float(score)})

print("Query:", q_demo)
print()
print("=" * 60)
print("VERSE-LEVEL retrieval (full verse as unit)")
print("=" * 60)
for h in verse_hits:
    print("[" + h["verse_id"] + "] score=" + str(round(h["score"], 4)))
    print(" ", h["text"][:140])
    print()

print("=" * 60)
print("CHUNK-LEVEL retrieval (sentence chunks of size", CHUNK_SIZE, ")")
print("=" * 60)
for h in chunk_hits:
    print("[" + h["verse_id"] + "] chunk=" + h["chunk_id"] + " score=" + str(round(h["score"], 4)))
    print(" ", h["text"][:140])
    print()

print("Observation:")
print(" Verse-level returns the full verse (may include off-topic sentences).")
print(" Chunk-level returns the most relevant sentence window within the verse.")
print(" Chunk scores are often higher because the chunk is more focused.")

## Section 6 â€” Contextual Compression

### The noise problem

Even after good retrieval, a passage may contain sentences irrelevant to the query. Every extra sentence in the prompt:
- Wastes LLM context window tokens
- Can distract the LLM, leading to answers that mix relevant and irrelevant content
- Increases hallucination risk when the LLM tries to synthesize noisy input

### Contextual compression approach

After retrieval, for each passage:
1. Split the passage into individual sentences
2. Score each sentence against the query using the cross-encoder
3. Keep only the top `N` sentences (e.g. top-2)
4. Replace the passage with this compressed version

This is a lightweight, model-based filter that strips noise while preserving the most relevant content. It does not require a separate compression model â€” we reuse the already-loaded cross-encoder.

In [ ]:
def compress_context(query, passages, top_n_sentences=2):
    compressed = []
    for p in passages:
        sentences = sent_tokenize(p["text"])
        if len(sentences) <= 1:
            compressed.append(p)
            continue
        # Score each sentence against the query
        pairs = [(query, s) for s in sentences]
        sent_scores = cross_encoder.predict(pairs)
        ranked_sents = sorted(zip(sent_scores, sentences), reverse=True)
        top_sents = [s for _, s in ranked_sents[:top_n_sentences]]
        new_p = dict(p)
        new_p["text"] = " ".join(top_sents)
        new_p["compressed"] = True
        compressed.append(new_p)
    return compressed

# Apply compression to chunk hits
compressed_hits = compress_context(q_demo, chunk_hits)
print("Contextual compression for:", q_demo)
print()
for orig, comp in zip(chunk_hits, compressed_hits):
    print("[" + orig["verse_id"] + "]")
    print("  Before:", len(orig["text"]), "chars |", orig["text"][:80])
    print("  After :", len(comp["text"]), "chars |", comp["text"][:80])
    print()

## Section 7 â€” RAGAS-style Evaluation

### Why do we need metrics?

Eyeballing answers is not scalable. Two failure modes:
- **Hallucination**: the answer makes claims not supported by the retrieved context
- **Irrelevance**: the answer is factually grounded but does not address the question

### RAGAS metrics (simplified)

**Faithfulness**: For each sentence in the answer, check whether it is entailed by at least one context passage. Score = fraction of answer sentences that are supported.

**Answer Relevance**: Does the answer actually address the question? Score = cross-encoder score of (query, answer) pair.

### Proxy model

Full RAGAS uses an LLM (e.g. GPT-4) as the judge. Here we use the cross-encoder as a proxy NLI (natural language inference) model â€” it scores how well a premise supports a hypothesis, which approximates entailment. This is a lightweight, offline, reproducible approximation.

### Important: interpreting CE scores

Cross-encoder scores are **raw logits** — unbounded real numbers, not probabilities. A score of 3.5 does not mean "35% relevant". What matters is **relative ordering**: a passage scoring 3.5 is more relevant than one scoring 0.4 for the same query. When comparing across queries, scores are not directly comparable. For the faithfulness metric we use the max CE score across context passages — a high max means at least one passage strongly supports that answer sentence.

In [ ]:
SYSTEM_PROMPT = (
    "You are a careful assistant. "
    "Use ONLY the context passages below to answer the question. "
    "Cite the verse IDs (e.g. [4:36]) when you make a claim. "
    "If the context does not contain the answer, say: "
    "'I cannot tell from the provided context.'"
)

def build_prompt(query, hits):
    blocks = []
    for h in hits:
        text = h["text"]
        if len(text) > VERSE_MAX_CHARS:
            text = text[:VERSE_MAX_CHARS] + "..."
        blocks.append("[" + h["verse_id"] + "] " + text)
    context = "\n\n".join(blocks)
    if len(context) > CONTEXT_MAX_CHARS:
        context = context[:CONTEXT_MAX_CHARS] + "\n\n[... truncated ...]"
    return SYSTEM_PROMPT + "\n\n--- Context ---\n" + context + "\n\n--- Question ---\n" + query + "\n\n--- Answer ---\n"

def generate(prompt):
    if gen_model is None:
        return "[skipped]"
    inputs = gen_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(DEVICE)
    with torch.no_grad():
        out = gen_model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, num_beams=NUM_BEAMS, early_stopping=True)
    return gen_tokenizer.decode(out[0], skip_special_tokens=True)

def faithfulness_score(answer, context_passages):
    if not answer or answer == "[skipped]":
        return 0.0
    sentences = sent_tokenize(answer) if answer else []
    if not sentences:
        return 0.0
    scores = []
    for sent in sentences:
        pairs = [(p["text"], sent) for p in context_passages]
        ce_scores = cross_encoder.predict(pairs)
        scores.append(float(max(ce_scores)))
    return round(sum(scores) / len(scores), 4)

def answer_relevance_score(query, answer):
    if not answer or answer == "[skipped]":
        return 0.0
    return round(float(cross_encoder.predict([(query, answer)])), 4)

# Evaluate on demo query
advanced_prompt = build_prompt(q_demo, reranked_hits)
advanced_answer = generate(advanced_prompt)
naive_prompt    = build_prompt(q_demo, [{"verse_id": vid, "text": df[df["verse_id"]==vid]["daryabadi"].values[0]} for vid in dense_ids])
naive_answer    = generate(naive_prompt)

print("Query:", q_demo)
print()
print("Naive answer  :", naive_answer)
print("Advanced answer:", advanced_answer)

In [ ]:
# Context passages for scoring
advanced_ctx = reranked_hits
naive_ctx = [{"verse_id": vid, "text": df[df["verse_id"]==vid]["daryabadi"].values[0]}
             for vid in dense_ids]

adv_faith = faithfulness_score(advanced_answer, advanced_ctx)
adv_relev = answer_relevance_score(q_demo, advanced_answer)
naive_faith = faithfulness_score(naive_answer, naive_ctx)
naive_relev = answer_relevance_score(q_demo, naive_answer)

print("=== RAGAS-style Evaluation ===")
print()
print("               Faithfulness   Relevance")
print("Naive RAG    :   " + str(naive_faith) + "         " + str(naive_relev))
print("Advanced RAG :   " + str(adv_faith) + "         " + str(adv_relev))

## Section 8 â€” Full Pipeline Comparison

Now we run both the naive (dense-only, no re-ranking) and advanced (hybrid + cross-encoder re-rank) pipelines on three different test queries. This gives us a more reliable picture of whether the improvements hold across topics, not just the demo query.

In [ ]:
test_queries = [
    "mercy and forgiveness for believers",
    "patience and gratitude in times of hardship",
    "justice and rights of the poor",
]

results_table = []
for q in test_queries:
    # Naive pipeline
    n_hits = [{"verse_id": vid, "text": df[df["verse_id"]==vid]["daryabadi"].values[0]}
               for vid in dense_retrieve(q)]
    n_ans = generate(build_prompt(q, n_hits))
    n_f = faithfulness_score(n_ans, n_hits)
    n_r = answer_relevance_score(q, n_ans)

    # Advanced pipeline
    a_pool = hybrid_retrieve(q, top_k=RERANK_POOL)
    a_hits = rerank(q, a_pool)
    a_ans = generate(build_prompt(q, a_hits))
    a_f = faithfulness_score(a_ans, a_hits)
    a_r = answer_relevance_score(q, a_ans)

    results_table.append({
        "Query": q[:40],
        "Naive Faith": n_f,
        "Adv Faith": a_f,
        "Naive Relev": n_r,
        "Adv Relev": a_r,
    })

results_df = pd.DataFrame(results_table)
print(results_df.to_string(index=False))

## Section 9 â€” What We Built and What's Still Missing

### Summary of upgrades

| Technique | What it fixes | Where introduced |
|---|---|---|
| Hybrid retrieval (BM25 + RRF) | Dense-only misses exact-match terms | Section 2 |
| Cross-encoder re-ranking | Bi-encoder scores query/doc independently | Section 3 |
| HyDE query expansion | Short sparse queries miss paraphrases | Section 4 |
| Sentence chunking with overlap | Full-verse granularity includes noise | Section 5 |
| Contextual compression | Retrieved passages contain irrelevant sentences | Section 6 |
| RAGAS-style eval (faithfulness + relevance) | No systematic quality measurement | Section 7 |

### What is still missing

| Gap | Description |
|---|---|
| Multi-hop reasoning | Questions requiring combining facts from multiple passages |
| Conversation memory | No ability to follow up on a previous question in context |
| Streaming | Generating the answer token-by-token for a responsive UI |
| Safety filters | No guardrails against harmful queries or toxic outputs |
| Production caching | Re-encoding the corpus on every notebook run is expensive |

## Reflection Questions

1. **RRF constant k**: RRF uses `k=60` as a smoothing constant. What happens if you set `k=1`? What about `k=1000`? Try to reason about the effect on rank compression before experimenting.

2. **Cross-encoder accuracy**: The cross-encoder performs better than the bi-encoder for re-ranking despite being slower. What architectural property allows it to model richer query-document interactions?

3. **HyDE failure modes**: HyDE assumes the LLM can generate a useful hypothetical document. Describe at least two scenarios where HyDE would perform *worse* than vanilla dense retrieval.

4. **Aggressive compression**: Contextual compression reduces noise by removing low-scoring sentences. What could go wrong if the compression step is too aggressive (e.g. keeps only 1 sentence per passage)? Consider both recall and coherence.

5. **Evaluation circularity**: In this notebook, faithfulness and answer relevance are both proxied by the same cross-encoder model that was also used in the re-ranking step. What are the methodological limitations of using the same model for both retrieval scoring and evaluation? How would you design a more rigorous evaluation?